# LLM 라벨링 및 앵커링(Zero-shot vs Few-shot 층화) 실험 분석

이 노트북은 파일럿 표본(40건)에 대한 **Zero-shot 3회 반복 일관성** 및 **Few-shot 층화(20풀 vs 23풀)** 실험 결과를 분석하고 비교합니다.

- **모델**: Claude Sonnet 5 (`effort=medium`, `thinking=adaptive`, 스키마 v4.0.0, 프롬프트 v5)
- **관련 의사결정**: 결정 13~14 (층화 인출), 결정 19~22 (가격 기준 스키마 및 앵커 풀), 결정 23~24 (반복 일관성 및 풀 확충)

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from scripts.labeling.analyze_consistency import load_runs as load_rep_runs
from scripts.labeling.compare_pools import main as run_pool_comparison
from scripts.labeling.run_claude_labeling import build_parser, make_manifest

In [ ]:
# 1. 3회 반복 일관성 (Zero-shot vs Few-shot 층화) 집계 실행
from scripts.labeling.analyze_consistency import main as analyze_rep_consistency
analyze_rep_consistency()

In [ ]:
# 2. 앵커 풀 확충 전/후 (20건 풀 vs 23건 풀) A/B 비교 및 전이 행렬
run_pool_comparison()

In [ ]:
# 3. 동적 층화 퓨샷 프롬프트 생성 dry-run 예시 확인
args = build_parser().parse_args([
    '--strategy', 'fewshot-stratified',
    '--anchor-pool', str(ROOT / 'data' / 'anchors' / 'anchor_pool_v1.jsonl'),
    '--limit', '2'
])
from scripts.labeling.run_claude_labeling import load_samples, build_retriever, _anchor_preview
samples = load_samples(ROOT / 'data' / 'samples' / 'labeling_pilot_sample_v0.1.0.jsonl', require_document_id=True)[:2]
retriever, meta = build_retriever(args)
print(_anchor_preview(samples, retriever, args))